In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
while project_root.name != 'python' and project_root.parent != project_root:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Downloading the data

In [ ]:
from data import download_tickers_history

TRADING_DAYS_PER_YEAR = 252

# set the date range for the historic data
start_date = datetime(year=2022, month=1, day=1)
end_date = datetime(year=2026, month=1, day=1)
tickers = ['NVDA', 'AAPL', 'PLTR', 'DKNG', 'CAT', 'INTC', 'AMZN', 'TSLA', 'GOOG', 'MSFT']
history = download_tickers_history(start_date, end_date, tickers)

history.head()


## The essence of the approach

The Ledoit-Wolf shrinkage method is needed to refine the noisy evaluation of covariate matrix $\Sigma$: with a large number of assets $N$ relative to the sample length of $T$, the sample covariate matrix is either warped or contains a lot of noise.

The sample covariance matrix $S = \frac{1}{T-1} X^T X$ is unbalanced but has a high error dispersion at $N / T = const$. If we invert $S$ in the the _GMF_ optimizer, small eigenvalues (noise) are inflated, distorting the weights of the portfolio.

### Method idea:
Squeeze a noisy sample matrix $S$ toward a structured, stable target matrix $F$:
$$\Sigma_{\text{LW}} = (1 - \delta) S + \delta F,$$
where:
* $S$ - sample covariance matrix;
* $F$ - structured target matrix. It is often used Constant Correlation Model, where the mean variance is retained and all pair correlations are replaced by the sample's mean correlation, or a diagonal matrix (zero correlations);
* $\delta \in [0, 1]$ - shrinkage intensity, which can be found analitically by minimizing Mean Square Error in Frobenius norm $\min_{\delta} E\left[ \Vert{}\Sigma_{\text{LW}} - \Sigma_{\text{true}}\Vert{}_F^2 \right]$, and has a closed-form solution $\delta^*$ :$$\delta^* = \frac{\text{Variance of the Sample Covariances } S}{\text{Squared Distance between sample } S \text{ and target matrix } F}$$

In [ ]:
from numpy import linalg as lg
from data.processors import log_returns

# Sample covariance matrix
returns = log_returns(history)
sample_cov = returns.cov() * TRADING_DAYS_PER_YEAR # type: ignore

# Build structured target matrix (FIX)
num_assets = sample_cov.columns.nunique()
rho = np.diag(sample_cov).mean()
target_matrix = np.diag(np.diag(sample_cov))
for i in range(0, num_assets):
    for j in range(0, num_assets):
        if i != j:
            target_matrix[i, j] = rho

# Find Shrinkage intencity coefficient
shrinkage_intensity = sample_cov.std().sum() / lg.norm(sample_cov - target_matrix) ** 2

cov_shrunk = (1 - shrinkage_intensity) * sample_cov + shrinkage_intensity * target_matrix
cov_shrunk


Now find the shrinkage_intencity and the shrunk assets covariance matrix via `sklearn` tools

In [ ]:
from sklearn.covariance import LedoitWolf

returns = np.asarray(log_returns(history))
lw = LedoitWolf()
lw.fit(returns)

cov_shrunk = lw.covariance_ * TRADING_DAYS_PER_YEAR
shrinkage_intensity = lw.shrinkage_ # optimal delta
cov_shrunk


#### GMF Optimization with Ledouit-Wolf shrinkage model and data visualization

Find the Efficient Frontier distribution

In [ ]:
from src.portfolio import optimize_portfolio, get_risk_free_rate

optimum_df = optimize_portfolio(tickers_df=history, rf_base='T_BILLS')
risk_free_rate = get_risk_free_rate('T_BILLS', start_date, end_date)
risk_free_rate

Find the max Sharpe

In [ ]:
from src.portfolio import find_max_sharpe

(max_sharpe, stocks_w) = find_max_sharpe(
    tickers_df=history,
    rf_base='T_BILLS',
    opt_models=['LEDOIT_WOLF'])

exact_max_ret = max_sharpe.tangency_return
exact_max_vol = max_sharpe.tangency_vol
exact_max_sharpe = max_sharpe.max_sharpe

print(f"Exact Max Sharpe Ratio: {max_sharpe.max_sharpe:.4f}")
print(f"Exact Tangency Return: {exact_max_ret:.2%}")
print(f"Exact Tangency Volatility: {exact_max_vol:.2%}")

print("Exact optimum stocks distribution:")
stocks_w

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

vol = optimum_df['vol']
sharpe = optimum_df['sharpe']
returns = optimum_df['return']
max_vol = vol.max()
max_sharpe = sharpe.max()
max_sharpe_ind = sharpe.idxmax()

# capital distribution line
cal_x = np.linspace(0, max_vol, 100)
cal_y = max_sharpe * cal_x + risk_free_rate

ax.plot(
    cal_x,
    cal_y,
    color='darkblue',
    alpha=0.7,
    linewidth=2, 
)
ax.spines['left'].set_position('zero')
x_label = max_vol * 0.85
y_label = max_sharpe * x_label + risk_free_rate

# Add text on the plot
ax.text(
    x_label,
    y_label + 0.02,
    'Capital Market Line',
    color='darkblue',
    fontweight='bold'
)

# mark risk free rate point
plt.plot(0, risk_free_rate, marker="o", markersize=8, markeredgecolor="red", markerfacecolor="yellow")
plt.annotate(
    'Risk free rate',
    xy=(0, risk_free_rate),
    xytext=(risk_free_rate + 0.01, risk_free_rate + 0.2),
    arrowprops=dict(facecolor='black', shrink=0.05)
)

# effective frontier
ef_x = vol
ef_y = returns
ax.plot(
    ef_x,
    ef_y,
    color='darkgreen',
    alpha=0.7,
    linewidth=2,
)

x_label = max_vol * 0.85
y_label = returns.max() * 0.85
# Add text on the plot
ax.text(
    x_label,
    y_label + 0.02,
    'Efficient frontier',
    color='darkgreen',
    fontweight='bold'
)

# mark the tangency portfolio point
plt.plot(exact_max_vol, exact_max_ret, marker="*", markersize=8, markerfacecolor="red")
plt.annotate(
    'Tangency portfolio',
    xy=(exact_max_vol, exact_max_ret),
    xytext=(exact_max_vol + 0.01, exact_max_ret - 0.05),
    arrowprops=dict(facecolor='black', shrink=0.02)
)

plt.title('Efficient Frontier')
plt.xlabel('Portfolio deviation', fontsize=12)
plt.ylabel('Expected return', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()
